<a href="https://colab.research.google.com/github/tassid/relacoes_fisicas_ambiente/blob/main/codigos_python/trabalho_aula_01_sudoeste.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trabalho da Aula 01 · Área de estudo do grupo
### Pato Branco, Francisco Beltrão e Dois Vizinhos (Sudoeste do Paraná)

**Disciplina:** Relações Físicas do Ambiente Agrícola (PPGTCA/UTFPR)

| Item pedido | Seção |
|---|---|
| Ambiente configurado, com o projeto de cada integrante | 1 |
| Área de estudo e justificativa agronômica | 2 |
| Mapa interativo com o limite da área | 3 |
| Área total, altitude média e chuva (2000–2024) | 4 |
| Pergunta agroclimática do grupo (e três sobre El Niño) | 5 |

## 1. Ambiente configurado
___

Cada integrante usa o **próprio projeto** do Google Cloud registrado no Earth Engine. Para rodar, troque `QUEM_ESTA_RODANDO` pelo seu nome; o notebook usa o ID correspondente.

> ⚠️ Se aparecer *"Google Earth Engine API has not been used in project ... or it is disabled"*, a API ainda não foi ativada no projeto. Abra o link que aparece no erro, clique em **Ativar** e rode a célula de novo depois de alguns minutos.

In [3]:
!pip install -q earthengine-api geemap

In [4]:
import ee
import geemap
import pandas as pd

# ID do projeto do Google Cloud de cada integrante
PROJETOS = {
    'Tassiane': 'ppgtca-relacoes-fisicas',
    'Eduardo':  'eduproject-510923',
    # 'Nome': 'id-do-projeto',   # demais integrantes
}

QUEM_ESTA_RODANDO = 'Tassiane'
ID_PROJETO = PROJETOS[QUEM_ESTA_RODANDO]

ee.Authenticate()
ee.Initialize(project=ID_PROJETO)
print(ee.String(f'Earth Engine conectado! ({QUEM_ESTA_RODANDO}: {ID_PROJETO})').getInfo())

Earth Engine conectado! (Tassiane: ppgtca-relacoes-fisicas)


## 2. Área de estudo
___

A área de estudo é formada por três municípios do **Sudoeste do Paraná**: **Pato Branco**, **Francisco Beltrão** e **Dois Vizinhos**. Eles estão no Terceiro Planalto Paranaense, sobre solos derivados de basalto, numa região de relevo ondulado e de transição entre os climas Cfa (verão quente) e Cfb (verão ameno), ambos subtropicais úmidos e sem estação seca definida.

**Justificativa agronômica**

- **Cultura predominante:** a agricultura é feita principalmente em propriedades familiares. O sistema típico é **soja ou milho no verão** e **trigo ou culturas de cobertura no inverno**. A região também é uma das principais bacias leiteiras do estado e tem forte produção de aves e suínos, que dependem do milho e das pastagens produzidos ali.
- **Importância regional:** Pato Branco, Francisco Beltrão e Dois Vizinhos são os polos do Sudoeste, com cooperativas, agroindústrias de carnes e leite e câmpus da UTFPR com cursos de agronomia e zootecnia. O clima desses municípios afeta tanto a produção de grãos quanto a de proteína animal da região.
- **Problema climático de interesse:** apesar de ser uma das regiões mais chuvosas do estado, o Sudoeste sofre com **estiagens** no verão, como na safra 2021/22, que afetam a soja, o milho e as pastagens. No inverno, as **geadas** são mais frequentes que no norte do estado e limitam o trigo e o milho semeado tarde. O **excesso de chuva** na colheita é outro risco.
- **Por que três municípios:** eles são próximos entre si, mas formam um gradiente de altitude, com Pato Branco no ponto mais alto. Isso permite comparar locais com clima parecido e testar se modelos calibrados em um município valem para os outros.

In [5]:
MUNICIPIOS = ['Pato Branco', 'Francisco Beltrão', 'Dois Vizinhos']
NOME_GAUL = {'Pato Branco': 'Pato Branco', 'Francisco Beltrão': 'Francisco Beltrao', 'Dois Vizinhos': 'Dois Vizinhos'}   # na base GAUL, sem acento
CORES = {'Pato Branco': 'red', 'Francisco Beltrão': 'blue', 'Dois Vizinhos': 'orange'}

gaul_pr = ee.FeatureCollection('FAO/GAUL/2015/level2').filter(ee.Filter.eq('ADM1_NAME', 'Parana'))

AREAS = {}
for m in MUNICIPIOS:
    AREAS[m] = gaul_pr.filter(ee.Filter.eq('ADM2_NAME', NOME_GAUL[m]))
    print(f'{m}: {AREAS[m].size().getInfo()} feição encontrada')

# A área de estudo é a união dos três municípios
area_estudo = gaul_pr.filter(ee.Filter.inList('ADM2_NAME', list(NOME_GAUL.values())))
geom_estudo = area_estudo.geometry()

/usr/local/lib/python3.13/dist-packages/ee/deprecation.py:215: DeprecationWarning: 

Attention required for FAO/GAUL/2015/level2! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by FAO/GAUL/2025/level2

Learn more: https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_2015_level2

  warnings.warn(warning, category=DeprecationWarning)


Pato Branco: 1 feição encontrada
Francisco Beltrão: 1 feição encontrada
Dois Vizinhos: 1 feição encontrada


## 3. Mapa interativo
___

Limites dos três municípios sobre o relevo (SRTM) e a chuva média anual de 2000 a 2024 (CHIRPS). Use o controle de camadas, no canto superior direito, para ligar e desligar cada camada.

In [6]:
ANO_INICIO_CHUVA, ANO_FIM_CHUVA = 2000, 2024  # período da chuva
N_ANOS_CHUVA = ANO_FIM_CHUVA - ANO_INICIO_CHUVA + 1

srtm = ee.Image('USGS/SRTMGL1_003')
chuva_periodo = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
                 .filterDate(f'{ANO_INICIO_CHUVA}-01-01', f'{ANO_FIM_CHUVA + 1}-01-01')
                 .sum())
chuva_media_anual = chuva_periodo.divide(N_ANOS_CHUVA)

mapa = geemap.Map()
mapa.add_basemap('HYBRID')
mapa.addLayer(srtm.clip(geom_estudo), {'min': 350, 'max': 950,
              'palette': ['006837', 'a6d96a', 'fee08b', 'd73027']}, 'Altitude (m)', False)
mapa.addLayer(chuva_media_anual.clip(geom_estudo), {'min': 1600, 'max': 2400,
              'palette': ['f7fbff', '6baed6', '08306b']}, 'Chuva média anual (mm/ano)', False)
for m in MUNICIPIOS:
    contorno = ee.Image().paint(AREAS[m], 0, 3)
    mapa.addLayer(contorno, {'palette': [CORES[m]]}, f'Limite de {m}')
mapa.centerObject(area_estudo, 9)
mapa

Map(center=[-26.012704981764372, -52.966455527338475], controls=(WidgetControl(options=['position', 'transpare…

## 4. Três informações sobre a área
___

Para cada município e para a área inteira:

- **Área total (km²):** área do polígono do município (FAO/GAUL).
- **Altitude média (m):** média do SRTM (30 m) dentro do limite.
- **Chuva total (mm):** soma das imagens diárias do CHIRPS de 2000 a 2024, média dentro do limite, e a **média anual** do período (total dividido por 25 anos).

In [7]:
def tres_informacoes(geom):
    area_km2 = geom.area().divide(1e6)
    altitude = srtm.reduceRegion(ee.Reducer.mean(), geom, scale=30, maxPixels=1e10).get('elevation')
    chuva_per = chuva_periodo.reduceRegion(ee.Reducer.mean(), geom, scale=5566, maxPixels=1e10).get('precipitation')
    return ee.Dictionary({'area': area_km2, 'altitude': altitude, 'chuva_per': chuva_per}).getInfo()


linhas = []
for nome, geom in [(m, AREAS[m].geometry()) for m in MUNICIPIOS] + [('Área de estudo (total)', geom_estudo)]:
    info = tres_informacoes(geom)
    linhas.append({'Município': nome,
                   'Área total (km²)': round(info['area'], 1),
                   'Altitude média (m)': round(info['altitude'], 0),
                   f'Chuva {ANO_INICIO_CHUVA}–{ANO_FIM_CHUVA} (mm)': round(info['chuva_per'], 0),
                   'Média anual (mm/ano)': round(info['chuva_per'] / N_ANOS_CHUVA, 0)})

tabela = pd.DataFrame(linhas).set_index('Município')
tabela

,Área total (km²),Altitude média (m),Chuva 2000–2024 (mm),Média anual (mm/ano)
Município,,,,
Pato Branco,540.1,709.0,48005.0,1920.0
Francisco Beltrão,719.0,656.0,49393.0,1976.0
Dois Vizinhos,418.9,528.0,47911.0,1916.0
Área de estudo (total),1677.9,641.0,48578.0,1943.0


> 💬 **Para discutir:** a altitude média do município é menor que a da sede urbana, porque os vales dos rios entram na média. O CHIRPS combina satélite e estações, com pixel de ~5,5 km; comparem a média anual com a da estação do INMET de cada cidade.

## 5. Pergunta agroclimática do grupo
___

> **No Sudoeste do Paraná, é possível estimar com precisão suficiente a demanda hídrica das culturas usando apenas a temperatura do ar e, com isso, explicar quanto da variação da produtividade da soja e do milho se deve ao déficit hídrico?**

Como a pergunta se encaixa nos projetos da disciplina:

| Projeto | Parte da pergunta |
|---|---|
| P1 · Elementos meteorológicos | Estimar a radiação solar e a ETo só com temperatura, e medir o erro dessa estimativa |
| P2 · Chuva, ETo e balanço hídrico | Usar essa ETo no balanço hídrico e identificar os períodos de deficiência nas safras de verão e de inverno |
| P3 · Produtividade | Penalizar a produtividade potencial pelo déficit hídrico (MZA-FAO) e comparar com a produtividade do IBGE |

### Perguntas complementares: El Niño-Oscilação Sul (ENOS)

O ENOS é a principal fonte de variabilidade climática de um ano para o outro no Sul do Brasil. Em geral, o **El Niño** está associado a mais chuva, principalmente na primavera, e a **La Niña** a estiagens, como na safra 2021/22. O Sudoeste do Paraná fica dentro dessa região de influência, então espera-se um sinal mais forte que no norte do estado.

1. **Chuva:** a chuva da primavera e do verão (outubro a março) nos três municípios é maior em anos de El Niño e menor em anos de La Niña, em comparação com anos neutros? A diferença é a mesma em Pato Branco, Francisco Beltrão e Dois Vizinhos?
2. **Radiação e ETo:** em anos de El Niño, a maior nebulosidade reduz a radiação solar global e a ETo? Os modelos de radiação por temperatura do P1 continuam funcionando bem nesses anos, ou o erro aumenta?
3. **Produtividade:** a produtividade da soja e do milho (IBGE) é menor nas safras de La Niña? Quanto dessa perda o déficit hídrico calculado pelo balanço hídrico (P2) consegue explicar?

Dados para responder: classificação dos anos pelo índice ONI da NOAA (El Niño, La Niña ou neutro), chuva do CHIRPS (desde 1981), radiação e ETo do BR-DWGD e produtividade da PAM/IBGE (SIDRA).